# Data Quality — Validation Job Notebook

**Run this notebook daily (schedule in Fabric Job Scheduler).**

This notebook:
1. Loads all active checks from `check_registry`
2. Executes the DAX expression for each model
3. Compares all models against the baseline (first model for each check)
4. Writes pass/fail results to `validation_results`
5. Shows a summary of failures (if any)

In [ ]:
# ── Configuration ────────────────────────────────────────────────────
from datetime import datetime
import uuid

LAKEHOUSE_NAME = "MyLakehouse"    # Your existing Lakehouse name
SCHEMA_NAME    = "data_quality"   # Schema where tables are stored

FULL_SCHEMA = f"{LAKEHOUSE_NAME}.{SCHEMA_NAME}"
RUN_DATE = datetime.now().date()
RUN_TIMESTAMP = datetime.now()
RUN_ID = str(uuid.uuid4())  # Unique ID for this run

print(f"Run ID: {RUN_ID}")
print(f"Run Date: {RUN_DATE}")

In [ ]:
# ── Load Active Checks ──────────────────────────────────────────────────
checks_df = spark.sql(f"""
SELECT check_name, model_name, workspace_name, dataset_name, dax_expression, run_frequency
FROM {FULL_SCHEMA}.check_registry
WHERE is_active = true
ORDER BY check_name, model_name
""").toPandas()

print(f"\n✓  Loaded {len(checks_df)} active check(s)")

# Load results that already exist for today (to handle crash recovery)
existing_results = spark.sql(f"""
SELECT check_name, model_name, run_frequency, status
FROM {FULL_SCHEMA}.check_registry r
INNER JOIN {FULL_SCHEMA}.validation_results v
  ON r.check_name = v.check_name AND r.model_name = v.model_name
WHERE v.run_date = '{RUN_DATE}'
""").toPandas()

skip_count = 0
if len(existing_results) > 0:
    print(f"  Skipping {len(existing_results)} check(s) already run today")
    skip_count = len(existing_results)

# Group by check_name to identify baselines
checks_by_name = checks_df.groupby("check_name")
baseline_models = checks_by_name.first().reset_index()
baseline_models.columns = ["check_name", "baseline_model_name", "baseline_workspace", "baseline_dataset", "baseline_dax", "baseline_frequency"]

print(f"  Baselines: {len(baseline_models)} check(s)")
print(f"  To process: {len(checks_df) - skip_count} check(s)")

In [ ]:
# ── Load Active Checks & Clean Previous Results ──────────────────────
checks_df = spark.sql(f"""
SELECT check_name, model_name, workspace_name, dataset_name, dax_expression
FROM {FULL_SCHEMA}.check_registry
WHERE is_active = true
ORDER BY check_name, model_name
""").toPandas()

print(f"\n✓  Loaded {len(checks_df)} active check(s)")

# Delete any results from today BEFORE this run (allows re-running same day)
# This makes the notebook idempotent
deleted = spark.sql(f"""
DELETE FROM {FULL_SCHEMA}.validation_results
WHERE run_date = '{RUN_DATE}' AND run_id != '{RUN_ID}'
""")

print(f"  Cleaned up any previous runs for {RUN_DATE}")

# Group by check_name to identify baselines
checks_by_name = checks_df.groupby("check_name")
baseline_models = checks_by_name.first().reset_index()
baseline_models.columns = ["check_name", "baseline_model_name", "baseline_workspace", "baseline_dataset", "baseline_dax"]

print(f"  Baselines: {len(baseline_models)} check(s)")

In [ ]:
# ── Execute DAX & Write Results Incrementally ───────────────────────
import sempy.fabric as sempy
from pyspark.sql.functions import lit
from datetime import datetime
import pandas as pd

pass_count = 0
fail_count = 0
error_count = 0
skipped_count = 0

# Pre-compute baseline values first
baselines = {}  # {check_name: value}
baseline_models_list = baseline_models.to_dict('records')

for b in baseline_models_list:
    check_name = b["check_name"]
    workspace_name = b["baseline_workspace"]
    dataset_name = b["baseline_dataset"]
    dax_expression = b["baseline_dax"]
    
    try:
        result_df = sempy.evaluate_dax(
            dataset=dataset_name,
            workspace=workspace_name,
            dax=dax_expression
        )
        if len(result_df) > 0:
            baselines[check_name] = float(result_df.iloc[0, 0])
        else:
            baselines[check_name] = None
    except Exception as e:
        print(f"  ⚠️  Baseline error for {check_name}: {str(e)}")
        baselines[check_name] = None

# Now process all checks, comparing to baselines
for _, row in checks_df.iterrows():
    check_name = row["check_name"]
    model_name = row["model_name"]
    workspace_name = row["workspace_name"]
    dataset_name = row["dataset_name"]
    dax_expression = row["dax_expression"]
    run_frequency = row["run_frequency"]
    
    # Check if this check already ran today
    already_ran = len(existing_results[
        (existing_results["check_name"] == check_name) & 
        (existing_results["model_name"] == model_name)
    ]) > 0
    
    # Skip if already ran AND frequency is ONCE_PER_DAY
    if already_ran and run_frequency == "ONCE_PER_DAY":
        print(f"  ⊘ {check_name:20} | {model_name:15} | SKIPPED (already ran today)")
        skipped_count += 1
        continue
    
    result_value = None
    error_msg = None
    status = "PASS"
    absolute_delta = None
    delta_pct = None
    baseline_value = baselines.get(check_name)
    
    try:
        # Execute DAX for this model
        result_df = sempy.evaluate_dax(
            dataset=dataset_name,
            workspace=workspace_name,
            dax=dax_expression
        )
        
        if len(result_df) == 0:
            result_value = None
            error_msg = "DAX returned empty result"
            status = "ERROR"
            error_count += 1
        else:
            # Extract the value
            result_value = float(result_df.iloc[0, 0])
            
            # Compare to baseline
            if baseline_value is not None and result_value is not None:
                absolute_delta = result_value - baseline_value
                if baseline_value != 0:
                    delta_pct = (absolute_delta / baseline_value) * 100
                    # Mark as FAIL if difference > 0.01%
                    if abs(delta_pct) > 0.01:
                        status = "FAIL"
                        fail_count += 1
                    else:
                        status = "PASS"
                        pass_count += 1
                else:
                    # Baseline is 0
                    if result_value != 0:
                        status = "FAIL"
                        fail_count += 1
                    else:
                        status = "PASS"
                        pass_count += 1
            else:
                status = "PASS"
                pass_count += 1
                
    except Exception as e:
        result_value = None
        error_msg = str(e)[:500]  # Truncate long errors
        status = "ERROR"
        error_count += 1
    
    # Write this result immediately to the table
    result_row = spark.createDataFrame([{
        "run_id": RUN_ID,
        "run_date": RUN_DATE,
        "run_timestamp": RUN_TIMESTAMP,
        "check_name": check_name,
        "model_name": model_name,
        "result_value": result_value,
        "baseline_model": baseline_models_list[0]["baseline_model_name"] if baseline_models_list else None,
        "baseline_value": baseline_value,
        "absolute_delta": absolute_delta,
        "delta_pct": delta_pct,
        "status": status,
        "error_message": error_msg
    }])
    
    result_row.write.format("DELTA").mode("append").option("mergeSchema", "true").saveAsTable(
        f"{FULL_SCHEMA}.validation_results"
    )
    
    # Print status line
    status_icon = "✓" if status == "PASS" else "✗" if status == "FAIL" else "!"
    print(f"  {status_icon} {check_name:20} | {model_name:15} | {status:6} | value={result_value}")

print(f"\n✓  Validation run complete")
print(f"    PASS:    {pass_count}")
print(f"    FAIL:    {fail_count}")
print(f"    ERROR:   {error_count}")
print(f"    SKIPPED: {skipped_count}")

## Summary

In [ ]:
from IPython.display import display

# Show failures from THIS run only
failures = spark.sql(f"""
SELECT check_name, model_name, result_value, baseline_value, delta_pct, status, error_message
FROM {FULL_SCHEMA}.validation_results
WHERE run_id = '{RUN_ID}' AND status != 'PASS'
ORDER BY check_name, model_name
""").toPandas()

if len(failures) > 0:
    print(f"\n⚠️  {len(failures)} failure(s) / error(s) in this run:\n")
    display(failures)
else:
    print(f"\n✅ No failures in this run")

print(f"\nRun ID: {RUN_ID}")
print(f"Completed at: {RUN_TIMESTAMP}")

## Table Maintenance

Automatic cleanup and optimization of Delta tables.

In [ ]:
import time

print("\n── Table Maintenance ──")

# OPTIMIZE validation_results — consolidate small files written incrementally
print(f"OPTIMIZE {FULL_SCHEMA}.validation_results...")
start = time.time()
spark.sql(f"OPTIMIZE {FULL_SCHEMA}.validation_results")
elapsed = time.time() - start
print(f"  ✓ Optimized in {elapsed:.2f}s")

# VACUUM validation_results — remove old file versions (retain 7 days)
print(f"VACUUM {FULL_SCHEMA}.validation_results...")
start = time.time()
spark.sql(f"VACUUM {FULL_SCHEMA}.validation_results RETAIN 7 DAYS")
elapsed = time.time() - start
print(f"  ✓ Vacuumed in {elapsed:.2f}s")

# ANALYZE TABLE — gather statistics for query optimization
print(f"ANALYZE {FULL_SCHEMA}.validation_results...")
start = time.time()
spark.sql(f"ANALYZE TABLE {FULL_SCHEMA}.validation_results COMPUTE STATISTICS")
elapsed = time.time() - start
print(f"  ✓ Analyzed in {elapsed:.2f}s")

print("\n✓  Maintenance complete")